In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import mne
from scipy.signal import welch

In [2]:
model = joblib.load("../models/random_forest_model.pkl")

print("Model Loaded Successfully!")

Model Loaded Successfully!


In [3]:
file_path = "../data/chb01/chb01_03.edf"
raw = mne.io.read_raw_edf(
    file_path,
    preload=True,
    verbose=False
)
print(raw)

C:\Users\Prajapati_Shivam\AppData\Local\Temp\ipykernel_27188\1579298447.py:2: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(


<RawEDF | chb01_03.edf, 23 x 921600 (3600.0 s), ~161.7 MiB, data loaded>


In [4]:
def preprocess_signal(raw):
    raw = raw.copy()
    raw.filter(
    l_freq=0.5,
    h_freq=40,
    verbose=False
    )
    return raw

In [5]:
processed_raw = preprocess_signal(raw)
print(processed_raw.info["sfreq"])

256.0


In [6]:
def create_windows(raw, window_size=4):
    data = raw.get_data()
    sfreq = raw.info["sfreq"]
    samples_per_window = int(window_size * sfreq)
    windows = []
    total_samples = data.shape[1]
    for start in range(0, total_samples, samples_per_window):
        end = start + samples_per_window
        if end <= total_samples:
            windows.append(data[:, start:end])
    return np.array(windows)

In [7]:
windows = create_windows(processed_raw) 
print("Total Window:", len(windows)) 
print("Shape:", windows[0].shape)

Total Window: 900
Shape: (23, 1024)


In [8]:
# Frequency bands used for feature extraction
frequency_bands = {
    "Delta": (0.5, 4),
    "Theta": (4, 8),
    "Alpha": (8, 13),
    "Beta": (13, 30),
    "Gamma": (30, 40)
}

def extract_statistical_features(window):
    # Calculate features for each EEG channel
    channel_means = np.mean(window, axis=1)
    channel_stds = np.std(window, axis=1)
    channel_variances = np.var(window, axis=1)
    
    # Average across all EEG channels
    mean_feature = np.mean(channel_means)
    std_feature = np.mean(channel_stds)
    variance_feature = np.mean(channel_variances)
    
    return mean_feature, std_feature, variance_feature
def extract_frequency_features(window, sfreq=256):
    channel_features = []
    for channel in window:
        frequencies, psd = welch(
            channel,
            fs=sfreq,
            nperseg=512
        )
        band_features = {}
        for band_name, (low_freq, high_freq) in frequency_bands.items():
            frequency_mask = (
                (frequencies >= low_freq) &
                (frequencies < high_freq)
            )
            band_power = np.trapezoid(
                psd[frequency_mask],
                frequencies[frequency_mask]
            )
            band_features[band_name] = band_power
        channel_features.append(band_features)
    channel_features_df = pd.DataFrame(channel_features)
    return channel_features_df.mean()

In [9]:
def extract_all_features(window, sfreq=256):
    
    # Statistical features
    mean_feature, std_feature, variance_feature = \
        extract_statistical_features(window)
    
    # Frequency-domain features
    frequency_features = extract_frequency_features(
        window,
        sfreq
    )
    
    # Combine all 8 features
    return [
        mean_feature,
        std_feature,
        variance_feature,
        frequency_features["Delta"],
        frequency_features["Theta"],
        frequency_features["Alpha"],
        frequency_features["Beta"],
        frequency_features["Gamma"]
    ]

In [10]:
# Extract features from all EEG windows

sfreq = processed_raw.info["sfreq"]

all_features = []

for i, window in enumerate(windows):
    
    features_for_window = extract_all_features(
        window,
        sfreq
    )
    
    all_features.append(features_for_window)

features_array = np.array(all_features)

print("Feature Extraction Completed!")
print("Feature Matrix Shape:", features_array.shape)

Feature Extraction Completed!
Feature Matrix Shape: (900, 8)


In [11]:
# Select one EEG window for prediction

window_index = 0

selected_features = features_array[window_index].reshape(1, -1)

print("Selected Window:", window_index)
print("Feature Shape:", selected_features.shape)


Selected Window: 0
Feature Shape: (1, 8)


In [12]:
# Extract features from all EEG windows

sfreq = processed_raw.info["sfreq"]

all_features = []

for i, window in enumerate(windows):
    
    features_for_window = extract_all_features(
        window,
        sfreq
    )
    
    all_features.append(features_for_window)

features_array = np.array(all_features)

print("Feature Extraction Completed!")
print("Feature Matrix Shape:", features_array.shape)

Feature Extraction Completed!
Feature Matrix Shape: (900, 8)


In [13]:
# Select one EEG window for prediction

window_index = 0

selected_features = features_array[window_index].reshape(1, -1)

print("Selected Window:", window_index)
print("Feature Shape:", selected_features.shape)

Selected Window: 0
Feature Shape: (1, 8)


In [14]:
# Predict the selected EEG window

prediction = model.predict(
    selected_features
)[0]

prediction_probability = model.predict_proba(
    selected_features
)[0]

if prediction == 0:
    result = "Normal / Non-Seizure"
else:
    result = "Seizure"

print("Prediction Result:", result)

print("\nPrediction Probabilities:")
print("Normal:", prediction_probability[0])
print("Seizure:", prediction_probability[1])

Prediction Result: Normal / Non-Seizure

Prediction Probabilities:
Normal: 0.98
Seizure: 0.02


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [15]:
# Test a known seizure window

seizure_window_index = 749

seizure_features = features_array[
    seizure_window_index
].reshape(1, -1)

seizure_prediction = model.predict(
    seizure_features
)[0]

seizure_probability = model.predict_proba(
    seizure_features
)[0]

if seizure_prediction == 0:
    seizure_result = "Normal / Non-Seizure"
else:
    seizure_result = "Seizure"

print("Seizure Window Index:", seizure_window_index)
print("Time Range: 2996–3000 seconds")

print("\nPrediction Result:", seizure_result)

print("\nPrediction Probabilities:")
print("Normal:", seizure_probability[0])
print("Seizure:", seizure_probability[1])

Seizure Window Index: 749
Time Range: 2996–3000 seconds

Prediction Result: Normal / Non-Seizure

Prediction Probabilities:
Normal: 1.0
Seizure: 0.0


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [16]:
# Test the known seizure window using threshold = 0.05

best_threshold = 0.05

seizure_probability_value = seizure_probability[1]

threshold_prediction = int(
    seizure_probability_value >= best_threshold
)

if threshold_prediction == 0:
    threshold_result = "Normal / Non-Seizure"
else:
    threshold_result = "Seizure"

print("Seizure Window Index:", seizure_window_index)
print("Time Range: 2996–3000 seconds")

print("\nSeizure Probability:", seizure_probability_value)
print("Selected Threshold:", best_threshold)

print("\nThreshold-Based Prediction:", threshold_result)

Seizure Window Index: 749
Time Range: 2996–3000 seconds

Seizure Probability: 0.0
Selected Threshold: 0.05

Threshold-Based Prediction: Normal / Non-Seizure


In [17]:
# Analyze all known seizure windows

seizure_window_indices = range(749, 759)

results = []

for i in seizure_window_indices:
    
    # Extract features
    window_features = features_array[i].reshape(1, -1)
    
    # Get prediction probability
    probability = model.predict_proba(
        window_features
    )[0, 1]
    
    # Default model prediction
    default_prediction = int(
        probability >= 0.50
    )
    
    # Threshold-adjusted prediction
    threshold_prediction = int(
        probability >= 0.05
    )
    
    # Calculate time range
    start_time = i * 4
    end_time = start_time + 4
    
    results.append({
        "Window": i,
        "Start Time (s)": start_time,
        "End Time (s)": end_time,
        "Seizure Probability": probability,
        "Default Prediction": default_prediction,
        "Threshold 0.05 Prediction": threshold_prediction
    })

seizure_results = pd.DataFrame(results)

print("Seizure Window Analysis:")
print(seizure_results)

Seizure Window Analysis:
   Window  Start Time (s)  End Time (s)  Seizure Probability  \
0     749            2996          3000                  0.0   
1     750            3000          3004                  0.0   
2     751            3004          3008                  0.0   
3     752            3008          3012                  0.0   
4     753            3012          3016                  0.0   
5     754            3016          3020                  0.0   
6     755            3020          3024                  0.0   
7     756            3024          3028                  0.0   
8     757            3028          3032                  0.0   
9     758            3032          3036                  0.0   

   Default Prediction  Threshold 0.05 Prediction  
0                   0                          0  
1                   0                          0  
2                   0                          0  
3                   0                          0  
4              

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.w

In [19]:
# Load labels for the train-test split

labels = np.load("../data/labels.npy")

print("Labels Shape:", labels.shape)

# Recreate the exact train-test split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features_array,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

print("\nTest set shape:", X_test.shape)

print("\nTest class distribution:")
print(np.unique(y_test, return_counts=True))

Labels Shape: (900,)

Test set shape: (180, 8)

Test class distribution:
(array([0, 1]), array([178,   2]))


In [20]:
# Find seizure samples inside the test set

test_seizure_indices = np.where(y_test == 1)[0]

print("Seizure positions inside test set:")
print(test_seizure_indices)

Seizure positions inside test set:
[ 70 104]


In [21]:
# Check seizure probabilities of the two test seizure samples

test_seizure_features = X_test[test_seizure_indices]

test_seizure_probabilities = model.predict_proba(
    test_seizure_features
)[:, 1]

print("Test seizure positions:", test_seizure_indices)

print(
    "Seizure probabilities:",
    test_seizure_probabilities
)

Test seizure positions: [ 70 104]
Seizure probabilities: [0. 0.]


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [22]:
# ============================================================
# FINAL PREDICTION SUMMARY
# ============================================================

print("=" * 60)
print("EEG SEIZURE DETECTION - FINAL PREDICTION SUMMARY")
print("=" * 60)

print("\n1. Normal EEG Window Test")
print("-" * 40)
print("Window Index:", 0)
print("Time Range: 0–4 seconds")
print("Prediction:", "Normal / Non-Seizure")
print("Normal Probability:", 0.98)
print("Seizure Probability:", 0.02)

print("\n2. Known Seizure EEG Window Test")
print("-" * 40)
print("Window Index:", 749)
print("Time Range: 2996–3000 seconds")
print("Actual Class:", "Seizure")
print("Model Prediction:", "Normal / Non-Seizure")
print("Seizure Probability:", 0.0)

print("\n3. All Known Seizure Windows")
print("-" * 40)
print("Window Range:", "749–758")
print("Total Seizure Windows Tested:", 10)
print("Detected as Seizure:", 0)
print("Detected as Normal:", 10)

print("\n4. Model Limitation")
print("-" * 40)
print(
    "The current Random Forest model fails to detect "
    "the known seizure windows in this recording."
)

print(
    "\nThis result demonstrates the impact of severe "
    "class imbalance on seizure detection performance."
)

print("\nPrediction Pipeline Completed Successfully!")

EEG SEIZURE DETECTION - FINAL PREDICTION SUMMARY

1. Normal EEG Window Test
----------------------------------------
Window Index: 0
Time Range: 0–4 seconds
Prediction: Normal / Non-Seizure
Normal Probability: 0.98
Seizure Probability: 0.02

2. Known Seizure EEG Window Test
----------------------------------------
Window Index: 749
Time Range: 2996–3000 seconds
Actual Class: Seizure
Model Prediction: Normal / Non-Seizure
Seizure Probability: 0.0

3. All Known Seizure Windows
----------------------------------------
Window Range: 749–758
Total Seizure Windows Tested: 10
Detected as Seizure: 0
Detected as Normal: 10

4. Model Limitation
----------------------------------------
The current Random Forest model fails to detect the known seizure windows in this recording.

This result demonstrates the impact of severe class imbalance on seizure detection performance.

Prediction Pipeline Completed Successfully!
